# Drought Monitoring in Limpopo River Basin (LRB)

* **Products used:**
[CHIRPS](https://digitaltwins.demos-only.iwmi.org/api2/chatbot/rainfall_data_channel)


### Background  

Drought monitoring plays a crucial role in understanding and mitigating the impacts of climate variability on water resources, agriculture, and ecosystems. This notebook focuses on visualizing drought conditions in the **Limpopo River Basin (LRB)** using the **Standardized Precipitation Index (SPI)**. The SPI index, a widely used drought indicator, is derived from **CHIRPS** (Climate Hazards Group InfraRed Precipitation with Station data) rainfall estimates, providing insights into precipitation anomalies over time.  

To achieve this, the notebook leverages powerful Python libraries such as **datacube, GeoPandas, pandas, Matplotlib, and Plotly**, enabling efficient data processing, statistical analysis, and visualization. By comparing CHIRPS rainfall data with ground station observations, the analysis helps assess the accuracy and reliability of satellite-derived precipitation estimates. The results from this study will aid researchers, hydrologists, and decision-makers in better understanding drought patterns, supporting informed water resource management and climate resilience planning.  

### Description

- Extract and process **CHIRPS rainfall data** for the Limpopo River Basin.  
- Calculate the **Standardized Precipitation Index (SPI)** to assess drought conditions.  
- Visualize **monthly drought patterns** and **rainfall distribution** using advanced plotting techniques.  



### Getting started

To run this analysis, run all the cells in the notebook, starting with the "Install packages" cell.

### Install libraries

In [ ]:

################# Uncomment below code lines #####################

# !pip install deafrica-tools

#!pip install pandas requests  numpy geopandas plotly


### Load libraries

In [ ]:
%matplotlib inline
import pandas as pd
import requests
import warnings
import datacube
import numpy as np
from deafrica_tools.plotting import display_map
import geopandas as gpd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

### Selection of CHIRPS rainfall station for drought monitoring



In this cell, we select the appropriate CHIRPS rainfall station based on the geographical region of interest (for example, the Lower River Basin). We use station metadata such as geographical coordinates and station IDs to filter the dataset. This selection is crucial for ensuring that the forecast analysis is representative of the target area.

### Key Steps:

- Identify the station with the best data quality and coverage.
- Filter the CHIRPS dataset to include data from the selected station only.

In [ ]:
def load_shapefile(file_path, crs=4326):
    """Load a shapefile and convert to WGS84 (EPSG:4326)."""
    gdf = gpd.read_file(file_path)
    gdf = gdf.to_crs(epsg=crs)
    gdf['lon'] = gdf.geometry.x
    gdf['lat'] = gdf.geometry.y
    return gdf.drop(columns='geometry')

def add_station_layer(fig, gdf, color, name, hover_cols):
    """Add a station layer to the figure with hover data."""
    hover_text = gdf.apply(lambda row: '<br>'.join([f"{col}: {row[col]}" for col in hover_cols]), axis=1)

    fig.add_trace(go.Scattermapbox(
        lon=gdf['lon'],
        lat=gdf['lat'],
        mode='markers',
        marker=dict(size=8, color=color, opacity=0.7),
        name=name,
        text=hover_text,  # Add formatted hover text
        hoverinfo="text"
    ))

# Load station data
ECMWF_stations = load_shapefile('data/chrips_rf_station.shp')

ECMWF_hover_columns = ['station', 'lat', 'lon']  # Add more columns if needed
# Create base map
fig = go.Figure()

# Add station layers with hover data
add_station_layer(fig, ECMWF_stations, color='red', name="ECMWF Stations", hover_cols=ECMWF_hover_columns)
# add_station_layer(fig, dws_stations, color='blue', name="DWA Stations", hover_cols=rain_hover_columns)

fig.update_layout(
    title="Overlay of CHIRPS Rainfall Stations",
    mapbox=dict(
        style="carto-positron",  # No token required
        center={"lat": -23.664, "lon": 29.609},
        zoom=5
    ),
    width=700,
    height=500
)

fig.show()

### User input : selection of AOI (CHIRPS station)

The user can select a CHIRPS station directly from the map to monitor drought conditions in the area surrounding the selected rain gauge.the rainfall data is fetched from the iwmi API. 

In [ ]:
CHIRPS_station= "s24846s25439e"

The **get_channel_id** function is used to identify the channel associated with the CHIRPS station. this data is required to fetch the rainfall data from the IWMI -api endpoint "https://digitaltwins.demos-only.iwmi.org/api2/chatbot/rainfall_data?station_name={station}&channel_id={channel_id}&start_date={start_date}&end_date={end_date} which requries channel id

In [ ]:
def get_channel_id(station_name):
    url_rf = "https://digitaltwins.demos-only.iwmi.org/api2/chatbot/rainfall_data_channel"
    
    try:
        response = requests.get(url_rf)
        response.raise_for_status()
        data = response.json()

        # You can collect all channel_ids matching the given station_name
        matching_channels = [entry["channel_id"] for entry in data if entry["station_name"] == station_name]

        if not matching_channels:
            return f"No channel ID found for station name: {station_name}"
        
        return matching_channels[0]  # could return multiple if station_name is repeated

    except requests.RequestException as e:
        return f"Error fetching data: {e}"

For the calculation of the drought index (SPI), 24 years of historical rainfall data are used to establish the general drought pattern. This piece of code retrieves the 24 years of rainfall data based on the user-selected station and channel.

In [ ]:
today = datetime.today().date()
first_date = datetime(datetime.now().year, 1, 1).strftime("%Y-%m-%d")
year_20_years_ago = datetime.now().year - 24
first_date_20 = datetime(year_20_years_ago, 1, 1).strftime("%Y-%m-%d")

print (f"Rainfall data retrival  from : {first_date_20}  end : {first_date}")


### Daily rainfall data extraction from selected stations

### Preparing Average Daily Climate and Comparing with Daily Forecast

This cell computes the extract daily rainfall for longer duration which is required to compute drought index (SPI) of historical rainfall data from the CHRIPS station. 

### Key Steps:
- Extract historical daily rainfall data from the CHRIPS station.
- Calculate the SPI INDEX.
- Overlay the monthly rainfall  with drought index.


In [ ]:
def fetch_url_notoekn(station,start_date,end_date,channel_id):
    
    url = f"https://digitaltwins.demos-only.iwmi.org/api2/chatbot/rainfall_data?station_name={station}&channel_id={channel_id}&start_date={start_date}&end_date={end_date}"
    print(url)
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        
        if df.empty:
            warnings.warn(f"No data available at the selected period ({start_date} to {end_date}) for station {station}")
            return None
        return df
    else:
        warnings.warn("Failed to fetch data from the API")
        return None

channel_id = get_channel_id(CHIRPS_station)
df = fetch_url_notoekn(CHIRPS_station,first_date_20,first_date,channel_id)
df_current = fetch_url_notoekn(CHIRPS_station,"2024-01-01","2024-12-30",channel_id)

### Standardized Precipitation Index (SPI)

The **Standardized Precipitation Index (SPI)** is a widely used drought index designed to quantify precipitation deficits or excesses over different time scales. It is particularly useful for assessing meteorological drought conditions by comparing observed precipitation with historical averages.

### Calculation of SPI
SPI is computed using the following steps:

1. **Data Collection**: Monthly or daily precipitation data over a long period (typically 30 years or more) is gathered.
2. **Fitting a Probability Distribution**: The precipitation data is fitted to a probability distribution, commonly a Gamma or Pearson Type III distribution.
3. **Transformation to Standard Normal Distribution**: The fitted data is then transformed into a standard normal distribution with a mean of zero and a standard deviation of one.
4. **SPI Computation**: The SPI value is derived from the standardized precipitation values.

### Interpretation of SPI Values
SPI values indicate how much precipitation deviates from the historical norm:

| SPI Value  | Drought Category          |
|------------|--------------------------|
| ≥ 2.0      | Extremely Wet             |
| 1.5 to 1.99 | Very Wet                 |
| 1.0 to 1.49 | Moderately Wet           |
| -0.99 to 0.99 | Near Normal            |
| -1.0 to -1.49 | Moderately Dry         |
| -1.5 to -1.99 | Severely Dry           |
| ≤ -2.0      | Extremely Dry            |

In [ ]:
def spi_index(df,df_current):
    
    from scipy.stats import gamma, norm
    import numpy as np

    mean_ppt = df['historical']
    history_ppt = df_current['historical']

    
    epsilon = 0.00001
    precipitation_2001_2022_adjusted = mean_ppt + epsilon
    his_ppt_2023_adjusted = history_ppt + epsilon
    shape, loc, scale = gamma.fit(precipitation_2001_2022_adjusted, floc=0)
    his_2023 = gamma.cdf(his_ppt_2023_adjusted, shape, loc, scale)
    spi_his = norm.ppf(his_2023)
    df_current['SPI_his'] = spi_his
    df_current[['date', 'forecast', 'historical','SPI_his']]
    return df_current

spi=spi_index(df,df_current)


### Visualization of drought index along with monthly rainfall

In [ ]:
# Convert 'date' to datetime format
spi["date"] = pd.to_datetime(spi["date"])

# Extract year and month
spi["year_month"] = spi["date"].dt.strftime("%m/%Y")  # Format as mm/yyyy
spi["month"] = spi["date"].dt.month

# Remove rows with NaN values in SPI_his and historical
df_filtered = spi.dropna(subset=["SPI_his", "historical"])

# Count the number of days per month
days_per_month = df_filtered["month"].value_counts().sort_index()

# Find months with complete data (29, 30, or 31 days)
complete_months = days_per_month[days_per_month.isin([29, 30, 31])].index

# Filter dataset to only include months with complete data
df_filtered = df_filtered[df_filtered["month"].isin(complete_months)]

# Compute the 90th percentile of SPI_his and the total monthly rainfall
monthly_stats = df_filtered.groupby("year_month").agg({
    "SPI_his": lambda x: x.quantile(0.90),
    "historical": "sum"
}).reset_index()

# Plotting
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot SPI_his 90th percentile
ax1.plot(monthly_stats["year_month"], monthly_stats["SPI_his"], marker="o", linestyle="-", color="b", label="90th Percentile SPI_his")
ax1.set_xlabel("Month (mm/yyyy)")
ax1.set_ylabel("SPI index 90th percentile", color="b")
ax1.tick_params(axis="y", labelcolor="b")

# Create second y-axis for rainfall
ax2 = ax1.twinx()
ax2.bar(monthly_stats["year_month"], monthly_stats["historical"], alpha=0.3, color="g", label="Total Monthly Rainfall")
ax2.set_ylabel("Total Monthly Rainfall (mm)", color="g")
ax2.tick_params(axis="y", labelcolor="g")

# Formatting
ax1.set_title("SPI historical & Monthly Rainfall")
ax1.set_xticks(range(len(monthly_stats["year_month"])))
ax1.set_xticklabels(monthly_stats["year_month"], rotation=45, ha="right")  
ax1.grid(True)

# Show plot
fig.tight_layout()
plt.show()


### Conclusion:
By retrieving 24 years of historical rainfall data for the selected CHIRPS station and associated channel, we calculated the Standardized Precipitation Index (SPI) to identify long-term drought patterns. Visualizing the SPI alongside monthly rainfall data provides a comprehensive view of both the intensity and frequency of droughts, helping to assess water availability trends and support drought monitoring and early warning efforts.

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

### Reference :  

- Vigneswaran, Kayathri; Ghosh, Surajit; Dickens, Chris; Garcia Andarcia, Mariangel. 2024. Experimental drought forecast for Limpopo River Basin. Colombo, Sri Lanka: International Water Management Institute (IWMI). CGIAR Initiative on Digital Innovation. 14p.https://hdl.handle.net/10568/158535
- Ghosh, S.; Vigneswaran, K.; Dickens,C.; Retief, H.; Garcia Andarcia, M.2024. A description of recent droughtprevalence in the Limpopo River Basin. Colombo, Sri Lanka: International Water Management Institute (IWMI). CGIAR Initiative on Digital Innovation. 13p.https://hdl.handle.net/10568/158254

📧 **Contact:**
- Kayathri Vigneswaran 
    *[v.kayathri@cgiar.org]*  
- Surajit Ghosh 
    *[s.ghosh@cgiar.org]*  